[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/085.%20The%20Uncapacitated%20Facility%20Location%20Problem/P85-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 85. The Uncapacitated Facility Location Problem
## Tier 3: The Advanced Algorithm (Hybrid SA-VNS Metaheuristic)

### Key assumptions
- Facilities have no capacity constraints
- Each customer must be assigned to exactly one facility
- Hybrid approach combines global exploration (Simulated Annealing) with local intensification (Variable Neighborhood Search)
- Temperature-controlled acceptance allows escaping local optima
- Multiple neighborhood structures provide thorough local search

### Approach (step-by-step)
1. **Initialize solution** and set parameters
   - Generate initial feasible solution using greedy heuristic
   - Set initial temperature and cooling schedule for SA
   - Define neighborhood structures for VNS

2. **Simulated Annealing phase** (global exploration)
   - Generate neighbor solutions using facility opening/closing operations
   - Accept worse solutions probabilistically based on temperature
   - Gradually cool down to reduce acceptance of worse solutions

3. **Variable Neighborhood Search phase** (local intensification)
   - Systematically explore different neighborhood structures
   - Apply local search to improve current solution
   - Shake operation to escape local minima

4. **Alternate between SA and VNS** phases
   - Use SA for global exploration and diversification
   - Use VNS for local intensification and refinement
   - Track best solution found during search

### What to look for in the results
- Temperature evolution and acceptance rate in SA phase
- Neighborhood exploration effectiveness in VNS phase
- Solution improvement through alternating phases
- Convergence behavior and final solution quality

### Concrete example (from the source)
8-facility, 15-customer instance demonstrating:
- Initial solution cost: 1250
- SA phase improvements with temperature control
- VNS phase local search refinements
- Final convergence after 2850 iterations

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for Hybrid SA-VNS Metaheuristic
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from dataclasses import dataclass
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(85)
random.seed(85)

print("Libraries imported successfully for Hybrid SA-VNS Metaheuristic")

📦 Packages Installed: 0
Sweep 100/1000: Energy = -6993.00, Best = -6993.00
Libraries imported successfully for Hybrid SA-VNS Metaheuristic


In [ ]:
@dataclass
class UFLPInstance:
    """Data class for Uncapacitated Facility Location Problem instance"""
    
    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    transport_costs: np.ndarray
    
    def __post_init__(self):
        """Validate data consistency"""
        assert len(self.fixed_costs) == self.n_facilities
        assert self.transport_costs.shape == (self.n_customers, self.n_facilities)

@dataclass
class UFLPSolution:
    """Data class for UFLP solution"""
    
    facilities_open: List[int]  # Binary list of facility opening decisions
    assignments: List[int]      # Facility index for each customer
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool

@dataclass
class SAParameters:
    """Parameters for Simulated Annealing"""
    
    initial_temp: float = 1000.0
    cooling_rate: float = 0.95
    min_temp: float = 1.0
    max_iterations: int = 1000

@dataclass
class VNSParameters:
    """Parameters for Variable Neighborhood Search"""
    
    max_neighborhoods: int = 3
    max_local_iterations: int = 100
    shake_intensity: int = 2

@dataclass
class HybridParameters:
    """Parameters for Hybrid SA-VNS"""
    
    sa_params: SAParameters
    vns_params: VNSParameters
    max_total_iterations: int = 3000
    alternation_frequency: int = 100  # Switch between SA and VNS every N iterations

print("Data structures defined for Hybrid SA-VNS")

Data structures defined for Hybrid SA-VNS


In [ ]:
try:
    def create_hybrid_example_instance():
        """Create the 8-facility, 15-customer instance from the source
        
        Returns:
            UFLPInstance: Problem instance for Hybrid SA-VNS demonstration
        """
        # Problem parameters from the concrete example
        n_facilities = 8
        n_customers = 15
        
        # Fixed costs for facilities (realistic values)
        fixed_costs = [120, 95, 140, 85, 110, 130, 100, 125]
        
        # Transportation costs (customers x facilities)
        # Create realistic cost matrix with some structure
        np.random.seed(43)  # For reproducible example
        base_costs = np.random.randint(15, 60, (n_customers, n_facilities))
        
        # Add some geographic structure (customers closer to certain facilities)
        for i in range(n_customers):
            # Each customer has 2-3 preferred facilities with lower costs
            preferred = np.random.choice(n_facilities, size=3, replace=False)
            for j in preferred:
                base_costs[i, j] = max(10, base_costs[i, j] - 20)
        
        transport_costs = base_costs
        
        return UFLPInstance(
            n_facilities=n_facilities,
            n_customers=n_customers,
            fixed_costs=fixed_costs,
            transport_costs=transport_costs
        )
    
    # Create the Hybrid SA-VNS example instance
    hybrid_instance = create_hybrid_example_instance()
    
    print(f"Created Hybrid SA-VNS UFLP instance:")
    print(f"  Facilities: {hybrid_instance.n_facilities}")
    print(f"  Customers: {hybrid_instance.n_customers}")
    print(f"  Fixed costs: {hybrid_instance.fixed_costs}")
    print(f"  Transport costs shape: {hybrid_instance.transport_costs.shape}")
    
    # Display summary statistics
    print(f"\nTransport cost statistics:")
    print(f"  Min: {np.min(hybrid_instance.transport_costs)}")
    print(f"  Max: {np.max(hybrid_instance.transport_costs)}")
    print(f"  Mean: {np.mean(hybrid_instance.transport_costs):.1f}")
    print(f"  Std: {np.std(hybrid_instance.transport_costs):.1f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
class HybridSAVNS:
    """Hybrid Simulated Annealing - Variable Neighborhood Search for UFLP"""
    
    def __init__(self, instance: UFLPInstance, params: HybridParameters):
        self.instance = instance
        self.params = params
        self.current_solution = None
        self.best_solution = None
        self.temperature = params.sa_params.initial_temp
        self.iteration = 0
        
        # Statistics tracking
        self.stats = {
            'best_costs': [],
            'current_costs': [],
            'temperatures': [],
            'acceptance_rates': [],
            'phase': [],  # 'SA' or 'VNS'
            'improvements': []
        }
    
    def generate_initial_solution(self) -> UFLPSolution:
        """Generate initial feasible solution using greedy heuristic
        
        Returns:
            UFLPSolution: Initial feasible solution
        """
        # Start with all facilities closed
        facilities_open = [False] * self.instance.n_facilities
        assignments = [-1] * self.instance.n_customers
        
        # Greedily open facilities based on average cost savings
        facility_savings = []
        for j in range(self.instance.n_facilities):
            avg_transport_cost = np.mean(self.instance.transport_costs[:, j])
            total_avg_cost = self.instance.fixed_costs[j] + avg_transport_cost * self.instance.n_customers
            facility_savings.append((j, total_avg_cost))
        
        # Sort by potential savings
        facility_savings.sort(key=lambda x: x[1])
        
        # Open facilities and assign customers
        for facility_idx, _ in facility_savings:
            # Check if opening this facility improves assignments
            improved = False
            for i in range(self.instance.n_customers):
                current_cost = (self.instance.transport_costs[i, assignments[i]] 
                               if assignments[i] >= 0 else float('inf'))
                new_cost = self.instance.transport_costs[i, facility_idx]
                if new_cost < current_cost:
                    assignments[i] = facility_idx
                    improved = True
            
            if improved:
                facilities_open[facility_idx] = True
        
        # Ensure all customers are assigned
        for i in range(self.instance.n_customers):
            if assignments[i] == -1:
                # Assign to cheapest open facility
                min_cost = float('inf')
                best_facility = -1
                for j in range(self.instance.n_facilities):
                    if facilities_open[j]:
                        cost = self.instance.transport_costs[i, j]
                        if cost < min_cost:
                            min_cost = cost
                            best_facility = j
                
                if best_facility >= 0:
                    assignments[i] = best_facility
                else:
                    # Open cheapest facility for this customer
                    min_fixed = float('inf')
                    best_facility = -1
                    for j in range(self.instance.n_facilities):
                        total_cost = self.instance.fixed_costs[j] + self.instance.transport_costs[i, j]
                        if total_cost < min_fixed:
                            min_fixed = total_cost
                            best_facility = j
                    
                    facilities_open[best_facility] = True
                    assignments[i] = best_facility
        
        # Calculate total cost
        fixed_cost = sum(self.instance.fixed_costs[j] 
                        for j in range(self.instance.n_facilities) if facilities_open[j])
        transport_cost = sum(self.instance.transport_costs[i, assignments[i]] 
                           for i in range(self.instance.n_customers))
        total_cost = fixed_cost + transport_cost
        
        return UFLPSolution(
            facilities_open=facilities_open,
            assignments=assignments,
            total_cost=total_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=True
        )
    
    def calculate_solution_cost(self, facilities_open: List[bool], assignments: List[int]) -> float:
        """Calculate total cost for given solution
        
        Args:
            facilities_open: Binary facility opening decisions
            assignments: Customer assignments
            
        Returns:
            Total cost
        """
        fixed_cost = sum(self.instance.fixed_costs[j] 
                        for j in range(self.instance.n_facilities) if facilities_open[j])
        transport_cost = sum(self.instance.transport_costs[i, assignments[i]] 
                           for i in range(self.instance.n_customers))
        return fixed_cost + transport_cost
    
    def assign_customers_to_facilities(self, facilities_open: List[bool]) -> List[int]:
        """Assign customers to cheapest open facilities
        
        Args:
            facilities_open: Binary facility opening decisions
            
        Returns:
            List of customer assignments
        """
        assignments = []
        for i in range(self.instance.n_customers):
            min_cost = float('inf')
            best_facility = -1
            for j in range(self.instance.n_facilities):
                if facilities_open[j]:
                    cost = self.instance.transport_costs[i, j]
                    if cost < min_cost:
                        min_cost = cost
                        best_facility = j
            assignments.append(best_facility)
        return assignments
    
    def sa_neighbor(self, solution: UFLPSolution) -> UFLPSolution:
        """Generate neighbor solution using Simulated Annealing operations
        
        Args:
            solution: Current solution
            
        Returns:
            Neighbor solution
        """
        new_facilities_open = solution.facilities_open.copy()
        
        # Random operation: open, close, or swap facility status
        operation = np.random.choice(['open', 'close', 'swap'])
        
        if operation == 'open':
            # Open a closed facility
            closed_facilities = [j for j, open in enumerate(new_facilities_open) if not open]
            if closed_facilities:
                facility_to_open = np.random.choice(closed_facilities)
                new_facilities_open[facility_to_open] = True
        
        elif operation == 'close':
            # Close an open facility (only if more than 1 is open)
            open_facilities = [j for j, open in enumerate(new_facilities_open) if open]
            if len(open_facilities) > 1:
                facility_to_close = np.random.choice(open_facilities)
                new_facilities_open[facility_to_close] = False
        
        else:  # swap
            # Swap status of two facilities
            if self.instance.n_facilities >= 2:
                fac1, fac2 = np.random.choice(self.instance.n_facilities, size=2, replace=False)
                new_facilities_open[fac1], new_facilities_open[fac2] = \
                    new_facilities_open[fac2], new_facilities_open[fac1]
        
        # Ensure at least one facility is open
        if not any(new_facilities_open):
            # Open facility with lowest fixed cost
            min_cost_idx = np.argmin(self.instance.fixed_costs)
            new_facilities_open[min_cost_idx] = True
        
        # Assign customers to new facility configuration
        new_assignments = self.assign_customers_to_facilities(new_facilities_open)
        
        # Calculate costs
        new_cost = self.calculate_solution_cost(new_facilities_open, new_assignments)
        fixed_cost = sum(self.instance.fixed_costs[j] 
                        for j in range(self.instance.n_facilities) if new_facilities_open[j])
        transport_cost = new_cost - fixed_cost
        
        return UFLPSolution(
            facilities_open=new_facilities_open,
            assignments=new_assignments,
            total_cost=new_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=True
        )
    
    def vns_neighborhood(self, solution: UFLPSolution, neighborhood_type: int) -> UFLPSolution:
        """Apply VNS neighborhood structure
        
        Args:
            solution: Current solution
            neighborhood_type: Type of neighborhood (1, 2, or 3)
            
        Returns:
            Solution after applying neighborhood
        """
        new_facilities_open = solution.facilities_open.copy()
        
        if neighborhood_type == 1:
            # Neighborhood 1: Toggle one facility
            facility_idx = np.random.randint(self.instance.n_facilities)
            new_facilities_open[facility_idx] = not new_facilities_open[facility_idx]
        
        elif neighborhood_type == 2:
            # Neighborhood 2: Toggle two facilities
            if self.instance.n_facilities >= 2:
                fac1, fac2 = np.random.choice(self.instance.n_facilities, size=2, replace=False)
                new_facilities_open[fac1] = not new_facilities_open[fac1]
                new_facilities_open[fac2] = not new_facilities_open[fac2]
        
        else:  # neighborhood_type == 3
            # Neighborhood 3: Toggle k facilities (k = shake_intensity)
            k = min(self.params.vns_params.shake_intensity, self.instance.n_facilities)
            facilities_to_toggle = np.random.choice(self.instance.n_facilities, size=k, replace=False)
            for fac in facilities_to_toggle:
                new_facilities_open[fac] = not new_facilities_open[fac]
        
        # Ensure at least one facility is open
        if not any(new_facilities_open):
            min_cost_idx = np.argmin(self.instance.fixed_costs)
            new_facilities_open[min_cost_idx] = True
        
        # Assign customers
        new_assignments = self.assign_customers_to_facilities(new_facilities_open)
        
        # Calculate costs
        new_cost = self.calculate_solution_cost(new_facilities_open, new_assignments)
        fixed_cost = sum(self.instance.fixed_costs[j] 
                        for j in range(self.instance.n_facilities) if new_facilities_open[j])
        transport_cost = new_cost - fixed_cost
        
        return UFLPSolution(
            facilities_open=new_facilities_open,
            assignments=new_assignments,
            total_cost=new_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=True
        )
    
    def vns_local_search(self, solution: UFLPSolution) -> UFLPSolution:
        """Apply VNS local search to improve solution
        
        Args:
            solution: Current solution
            
        Returns:
            Improved solution
        """
        current = solution
        improved = True
        
        while improved:
            improved = False
            
            # Try all neighborhood structures
            for k in range(1, self.params.vns_params.max_neighborhoods + 1):
                # Shake
                shaken = self.vns_neighborhood(current, k)
                
                # Local search in neighborhood k
                for _ in range(self.params.vns_params.max_local_iterations):
                    neighbor = self.vns_neighborhood(shaken, k)
                    
                    if neighbor.total_cost < shaken.total_cost:
                        shaken = neighbor
                    
                    # Accept improvement
                    if shaken.total_cost < current.total_cost:
                        current = shaken
                        improved = True
                        break
        
        return current
    
    def simulated_annealing_step(self, solution: UFLPSolution) -> Tuple[UFLPSolution, bool]:
        """Perform one step of Simulated Annealing
        
        Args:
            solution: Current solution
            
        Returns:
            Tuple of (new_solution, accepted)
        """
        # Generate neighbor
        neighbor = self.sa_neighbor(solution)
        
        # Calculate cost difference
        delta_cost = neighbor.total_cost - solution.total_cost
        
        # Accept or reject
        if delta_cost < 0:
            # Accept better solution
            return neighbor, True
        else:
            # Accept worse solution with probability exp(-delta_cost / temperature)
            if self.temperature > 0:
                acceptance_prob = np.exp(-delta_cost / self.temperature)
                if np.random.random() < acceptance_prob:
                    return neighbor, True
        
        return solution, False
    
    def update_temperature(self):
        """Update temperature using cooling schedule"""
        self.temperature = max(
            self.params.sa_params.min_temp,
            self.temperature * self.params.sa_params.cooling_rate
        )
    
    def solve(self) -> UFLPSolution:
        """Solve UFLP using Hybrid SA-VNS
        
        Returns:
            Best solution found
        """
        print("Starting Hybrid SA-VNS for UFLP...")
        print(f"Max iterations: {self.params.max_total_iterations}")
        print(f"Alternation frequency: {self.params.alternation_frequency}")
        
        # Generate initial solution
        self.current_solution = self.generate_initial_solution()
        self.best_solution = self.current_solution
        
        print(f"Initial solution cost: {self.current_solution.total_cost}")
        
        acceptance_count = 0
        total_attempts = 0
        
        while self.iteration < self.params.max_total_iterations:
            # Determine current phase
            phase = 'SA' if (self.iteration // self.params.alternation_frequency) % 2 == 0 else 'VNS'
            
            if phase == 'SA':
                # Simulated Annealing phase
                new_solution, accepted = self.simulated_annealing_step(self.current_solution)
                
                if accepted:
                    self.current_solution = new_solution
                    acceptance_count += 1
                    
                    # Update best solution
                    if new_solution.total_cost < self.best_solution.total_cost:
                        self.best_solution = new_solution
                
                total_attempts += 1
                
                # Update temperature
                self.update_temperature()
                
            else:  # VNS phase
                # Variable Neighborhood Search phase
                improved_solution = self.vns_local_search(self.current_solution)
                
                if improved_solution.total_cost < self.current_solution.total_cost:
                    self.current_solution = improved_solution
                    
                    # Update best solution
                    if improved_solution.total_cost < self.best_solution.total_cost:
                        self.best_solution = improved_solution
                
                acceptance_count += 1  # Count VNS improvements as acceptances
                total_attempts += 1
            
            # Record statistics
            acceptance_rate = acceptance_count / max(1, total_attempts)
            
            self.stats['best_costs'].append(self.best_solution.total_cost)
            self.stats['current_costs'].append(self.current_solution.total_cost)
            self.stats['temperatures'].append(self.temperature)
            self.stats['acceptance_rates'].append(acceptance_rate)
            self.stats['phase'].append(phase)
            self.stats['improvements'].append(
                self.current_solution.total_cost < self.stats['current_costs'][-2] if len(self.stats['current_costs']) > 1 else False
            )
            
            # Print progress
            if (self.iteration + 1) % 200 == 0 or self.iteration == 0:
                print(f"Iteration {self.iteration + 1}: {phase} Phase, "
                      f"Best = {self.best_solution.total_cost:.1f}, "
                      f"Current = {self.current_solution.total_cost:.1f}, "
                      f"Temp = {self.temperature:.1f}")
            
            self.iteration += 1
        
        print(f"Completed {self.iteration} iterations")
        print(f"Final best cost: {self.best_solution.total_cost}")
        
        return self.best_solution

print("Hybrid SA-VNS class defined")

Hybrid SA-VNS class defined


In [ ]:
try:
    # Solve the UFLP using Hybrid SA-VNS
    sa_params = SAParameters(
        initial_temp=1000.0,
        cooling_rate=0.95,
        min_temp=1.0,
        max_iterations=1000
    )
    
    vns_params = VNSParameters(
        max_neighborhoods=3,
        max_local_iterations=100,
        shake_intensity=2
    )
    
    hybrid_params = HybridParameters(
        sa_params=sa_params,
        vns_params=vns_params,
        max_total_iterations=3000,
        alternation_frequency=100
    )
    
    hybrid_solver = HybridSAVNS(hybrid_instance, hybrid_params)
    hybrid_solution = hybrid_solver.solve()
    
    print("\n=== HYBRID SA-VNS SOLUTION RESULTS ===")
    print(f"Total cost: {hybrid_solution.total_cost:.2f}")
    print(f"Fixed cost: {hybrid_solution.fixed_cost:.2f}")
    print(f"Transport cost: {hybrid_solution.transport_cost:.2f}")
    print(f"\nFacilities opened: {[j+1 for j, open in enumerate(hybrid_solution.facilities_open) if open]}")
    print(f"Number of facilities open: {sum(hybrid_solution.facilities_open)}")
    
    print("\nCustomer assignments:")
    for i, facility in enumerate(hybrid_solution.assignments):
        cost = hybrid_instance.transport_costs[i, facility]
        print(f"  Customer {i+1} -> Facility {facility+1} (Cost: {cost})")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'hybrid_instance' is not defined...]

In [ ]:
try:
    def visualize_hybrid_results(hybrid_solver: HybridSAVNS, solution: UFLPSolution):
        """Create comprehensive visualization of Hybrid SA-VNS results
        
        Args:
            hybrid_solver: Hybrid SA-VNS solver with statistics
            solution: Best solution found
        """
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Hybrid SA-VNS - UFLP Solution Analysis', fontsize=16, fontweight='bold')
        
        # 1. Cost convergence over iterations
        ax1 = axes[0, 0]
        iterations = range(1, len(hybrid_solver.stats['best_costs']) + 1)
        ax1.plot(iterations, hybrid_solver.stats['best_costs'], 'o-', linewidth=2, 
                markersize=4, color='#2E86AB', label='Best Cost', alpha=0.7)
        ax1.plot(iterations, hybrid_solver.stats['current_costs'], 's-', linewidth=1, 
                markersize=3, color='#A23B72', label='Current Cost', alpha=0.5)
        ax1.set_xlabel('Iteration')
        ax1.set_ylabel('Total Cost')
        ax1.set_title('Cost Convergence')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Temperature evolution (SA phase)
        ax2 = axes[0, 1]
        sa_iterations = [i for i, phase in enumerate(hybrid_solver.stats['phase']) if phase == 'SA']
        sa_temps = [hybrid_solver.stats['temperatures'][i] for i in sa_iterations]
        sa_iter_numbers = [i+1 for i in sa_iterations]
        
        if sa_iter_numbers:
            ax2.plot(sa_iter_numbers, sa_temps, 'o-', linewidth=2, markersize=4, color='#F18F01')
            ax2.set_xlabel('Iteration')
            ax2.set_ylabel('Temperature')
            ax2.set_title('Temperature Evolution (SA Phases)')
            ax2.set_yscale('log')
            ax2.grid(True, alpha=0.3)
        else:
            ax2.text(0.5, 0.5, 'No SA phases', ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title('Temperature Evolution (SA Phases)')
        
        # 3. Phase alternation and improvements
        ax3 = axes[1, 0]
        phases_numeric = [1 if phase == 'SA' else 2 for phase in hybrid_solver.stats['phase']]
        improvements = hybrid_solver.stats['improvements']
        
        # Plot phases as background
        ax3.fill_between(iterations, 0, phases_numeric, alpha=0.3, color='lightblue', label='Phase')
        
        # Plot improvements as points
        improvement_iterations = [i+1 for i, imp in enumerate(improvements) if imp]
        improvement_costs = [hybrid_solver.stats['current_costs'][i] for i, imp in enumerate(improvements) if imp]
        
        if improvement_iterations:
            ax3.scatter(improvement_iterations, improvement_costs, 
                       color='red', s=50, alpha=0.8, label='Improvements', zorder=5)
        
        ax3.set_xlabel('Iteration')
        ax3.set_ylabel('Cost / Phase')
        ax3.set_title('Phase Alternation and Improvements')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        ax3.set_yticks([1, 2])
        ax3.set_yticklabels(['SA', 'VNS'])
        
        # 4. Final facility configuration
        ax4 = axes[1, 1]
        facility_numbers = [f'F{j+1}' for j in range(hybrid_solver.instance.n_facilities)]
        opening_status = ['Open' if open_fac else 'Closed' for open_fac in solution.facilities_open]
        colors_fac = ['#F18F01' if open_fac else '#C73E1D' for open_fac in solution.facilities_open]
        
        bars = ax4.bar(facility_numbers, [1 if open_fac else 0] * hybrid_solver.instance.n_facilities, color=colors_fac)
        ax4.set_title('Final Facility Configuration')
        ax4.set_ylabel('Open (1) / Closed (0)')
        ax4.set_ylim(-0.1, 1.1)
        
        # Add fixed costs as text
        for i, (bar, open_fac, cost) in enumerate(zip(bars, solution.facilities_open, 
                                                     hybrid_solver.instance.fixed_costs)):
            ax4.text(bar.get_x() + bar.get_width()/2, 0.5, f'${cost}k', 
                    ha='center', va='center', fontweight='bold', fontsize=8)
        
        plt.tight_layout()
        plt.show()
        
        # Additional analysis
        print("\n=== HYBRID ALGORITHM ANALYSIS ===")
        
        # Phase statistics
        sa_count = hybrid_solver.stats['phase'].count('SA')
        vns_count = hybrid_solver.stats['phase'].count('VNS')
        total_improvements = sum(hybrid_solver.stats['improvements'])
        
        print(f"SA phases: {sa_count}, VNS phases: {vns_count}")
        print(f"Total improvements: {total_improvements}")
        print(f"Improvement rate: {total_improvements/len(hybrid_solver.stats['improvements'])*100:.1f}%")
        
        # Temperature statistics
        if sa_temps:
            print(f"Initial temperature: {sa_temps[0]:.1f}")
            print(f"Final temperature: {sa_temps[-1]:.1f}")
            print(f"Temperature reduction: {(1 - sa_temps[-1]/sa_temps[0])*100:.1f}%")
        
        # Solution quality
        initial_cost = hybrid_solver.stats['current_costs'][0]
        final_cost = hybrid_solution.total_cost
        improvement = ((initial_cost - final_cost) / initial_cost) * 100
        
        print(f"\nInitial cost: {initial_cost:.1f}")
        print(f"Final cost: {final_cost:.1f}")
        print(f"Total improvement: {improvement:.2f}%")
    
    # Visualize the Hybrid SA-VNS results
    visualize_hybrid_results(hybrid_solver, hybrid_solution)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'hybrid_solver' is not defined...]

In [ ]:
try:
    def compare_hybrid_with_baselines(instance: UFLPInstance, hybrid_solution: UFLPSolution):
        """Compare Hybrid SA-VNS with other solution methods
        
        Args:
            instance: UFLP instance
            hybrid_solution: Hybrid SA-VNS solution
        """
        def greedy_solution():
            """Simple greedy heuristic"""
            facilities_open = [False] * instance.n_facilities
            assignments = [-1] * instance.n_customers
            
            # Calculate average costs and sort
            avg_costs = []
            for j in range(instance.n_facilities):
                avg_transport = np.mean(instance.transport_costs[:, j])
                total_avg = instance.fixed_costs[j] + avg_transport * instance.n_customers
                avg_costs.append((j, total_avg))
            
            avg_costs.sort(key=lambda x: x[1])
            
            for j, _ in avg_costs:
                improved = False
                for i in range(instance.n_customers):
                    current_cost = (instance.transport_costs[i, assignments[i]] 
                                   if assignments[i] >= 0 else float('inf'))
                    new_cost = instance.transport_costs[i, j]
                    if new_cost < current_cost:
                        assignments[i] = j
                        improved = True
                
                if improved:
                    facilities_open[j] = True
            
            # Ensure all customers assigned
            for i in range(instance.n_customers):
                if assignments[i] == -1:
                    min_cost = float('inf')
                    best_facility = -1
                    for j in range(instance.n_facilities):
                        if facilities_open[j]:
                            cost = instance.transport_costs[i, j]
                            if cost < min_cost:
                                min_cost = cost
                                best_facility = j
                    
                    if best_facility >= 0:
                        assignments[i] = best_facility
                    else:
                        facilities_open[0] = True
                        assignments[i] = 0
            
            # Calculate costs
            fixed_cost = sum(instance.fixed_costs[j] 
                            for j in range(instance.n_facilities) if facilities_open[j])
            transport_cost = sum(instance.transport_costs[i, assignments[i]] 
                               for i in range(instance.n_customers))
            total_cost = fixed_cost + transport_cost
            
            return UFLPSolution(
                facilities_open=facilities_open,
                assignments=assignments,
                total_cost=total_cost,
                fixed_cost=fixed_cost,
                transport_cost=transport_cost,
                feasible=True
            )
        
        def random_solution():
            """Random solution for comparison"""
            best_random = None
            best_cost = float('inf')
            
            for _ in range(1000):
                facilities_open = [np.random.random() < 0.4 for _ in range(instance.n_facilities)]
                
                if not any(facilities_open):
                    facilities_open[np.random.randint(instance.n_facilities)] = True
                
                assignments = []
                for i in range(instance.n_customers):
                    min_cost = float('inf')
                    best_facility = -1
                    for j in range(instance.n_facilities):
                        if facilities_open[j]:
                            cost = instance.transport_costs[i, j]
                            if cost < min_cost:
                                min_cost = cost
                                best_facility = j
                    assignments.append(best_facility)
                
                fixed_cost = sum(instance.fixed_costs[j] 
                                for j in range(instance.n_facilities) if facilities_open[j])
                transport_cost = sum(instance.transport_costs[i, assignments[i]] 
                                   for i in range(instance.n_customers))
                total_cost = fixed_cost + transport_cost
                
                if total_cost < best_cost:
                    best_cost = total_cost
                    best_random = UFLPSolution(
                        facilities_open=facilities_open,
                        assignments=assignments,
                        total_cost=total_cost,
                        fixed_cost=fixed_cost,
                        transport_cost=transport_cost,
                        feasible=True
                    )
            
            return best_random
        
        # Get baseline solutions
        greedy_sol = greedy_solution()
        random_sol = random_solution()
        
        print("=== HYBRID SA-VNS VS BASELINE COMPARISON ===")
        
        solutions = [
            ("Hybrid SA-VNS", hybrid_solution),
            ("Greedy", greedy_sol),
            ("Random Best", random_sol)
        ]
        
        print(f"\n{'Method':<15} {'Total Cost':<12} {'Fixed':<10} {'Transport':<12} {'Facilities':<12}")
        print("-" * 70)
        
        for name, sol in solutions:
            print(f"{name:<15} {sol.total_cost:<12.1f} {sol.fixed_cost:<10.1f} "
                  f"{sol.transport_cost:<12.1f} {sum(sol.facilities_open):<12}")
        
        # Calculate improvements
        hybrid_cost = hybrid_solution.total_cost
        greedy_cost = greedy_sol.total_cost
        random_cost = random_sol.total_cost
        
        greedy_improvement = ((greedy_cost - hybrid_cost) / greedy_cost) * 100
        random_improvement = ((random_cost - hybrid_cost) / random_cost) * 100
        
        print(f"\nImprovement over Greedy: {greedy_improvement:.2f}%")
        print(f"Improvement over Random: {random_improvement:.2f}%")
        
        # Visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Cost comparison
        methods = [name for name, _ in solutions]
        total_costs = [sol.total_cost for _, sol in solutions]
        fixed_costs = [sol.fixed_cost for _, sol in solutions]
        transport_costs = [sol.transport_cost for _, sol in solutions]
        
        x = np.arange(len(methods))
        width = 0.25
        
        ax1.bar(x - width, total_costs, width, label='Total Cost', color='#2E86AB')
        ax1.bar(x, fixed_costs, width, label='Fixed Cost', color='#A23B72')
        ax1.bar(x + width, transport_costs, width, label='Transport Cost', color='#F18F01')
        
        ax1.set_xlabel('Solution Method')
        ax1.set_ylabel('Cost')
        ax1.set_title('Cost Comparison: Hybrid SA-VNS vs Baselines')
        ax1.set_xticks(x)
        ax1.set_xticklabels(methods)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Facility count and improvement
        facility_counts = [sum(sol.facilities_open) for _, sol in solutions]
        colors_fac = ['#2E86AB', '#A23B72', '#F18F01']
        
        ax2.bar(methods, facility_counts, color=colors_fac)
        ax2.set_xlabel('Solution Method')
        ax2.set_ylabel('Number of Facilities Open')
        ax2.set_title('Facility Count Comparison')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # Compare Hybrid SA-VNS with baseline methods
    compare_hybrid_with_baselines(hybrid_instance, hybrid_solution)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'hybrid_instance' is not defined...]

In [ ]:
try:
    def parameter_tuning_analysis(instance: UFLPInstance):
        """Analyze the effect of key parameters on Hybrid SA-VNS performance
        
        Args:
            instance: UFLP instance to analyze
        """
        print("=== PARAMETER TUNING ANALYSIS ===")
        
        # Test different cooling rates
        print("\n1. Cooling Rate Sensitivity:")
        cooling_rates = [0.90, 0.93, 0.95, 0.97, 0.99]
        cooling_results = []
        
        for rate in cooling_rates:
            sa_params = SAParameters(
                initial_temp=1000.0,
                cooling_rate=rate,
                min_temp=1.0,
                max_iterations=1000
            )
            
            hybrid_params = HybridParameters(
                sa_params=sa_params,
                vns_params=vns_params,
                max_total_iterations=1500,  # Reduced for faster analysis
                alternation_frequency=100
            )
            
            solver = HybridSAVNS(instance, hybrid_params)
            solution = solver.solve()
            
            cooling_results.append({
                'cooling_rate': rate,
                'cost': solution.total_cost,
                'facilities': sum(solution.facilities_open),
                'improvements': sum(solver.stats['improvements'])
            })
            
            print(f"  Rate {rate:.2f}: Cost={solution.total_cost:.1f}, Facilities={sum(solution.facilities_open)}, Improvements={sum(solver.stats['improvements'])}")
        
        # Test different alternation frequencies
        print("\n2. Alternation Frequency Sensitivity:")
        frequencies = [50, 100, 200, 300, 500]
        freq_results = []
        
        for freq in frequencies:
            hybrid_params = HybridParameters(
                sa_params=sa_params,
                vns_params=vns_params,
                max_total_iterations=1500,
                alternation_frequency=freq
            )
            
            solver = HybridSAVNS(instance, hybrid_params)
            solution = solver.solve()
            
            freq_results.append({
                'frequency': freq,
                'cost': solution.total_cost,
                'facilities': sum(solution.facilities_open),
                'improvements': sum(solver.stats['improvements'])
            })
            
            print(f"  Freq {freq}: Cost={solution.total_cost:.1f}, Facilities={sum(solution.facilities_open)}, Improvements={sum(solver.stats['improvements'])}")
        
        # Create parameter sensitivity plots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Cooling rate sensitivity
        rates = [r['cooling_rate'] for r in cooling_results]
        costs_cooling = [r['cost'] for r in cooling_results]
        improvements_cooling = [r['improvements'] for r in cooling_results]
        
        ax1.plot(rates, costs_cooling, 'o-', linewidth=2, markersize=8, color='#2E86AB')
        ax1.set_xlabel('Cooling Rate')
        ax1.set_ylabel('Final Cost', color='#2E86AB')
        ax1.tick_params(axis='y', labelcolor='#2E86AB')
        ax1.set_title('Cooling Rate Sensitivity')
        ax1.grid(True, alpha=0.3)
        
        ax1_twin = ax1.twinx()
        ax1_twin.plot(rates, improvements_cooling, 's--', linewidth=2, markersize=8, color='#A23B72')
        ax1_twin.set_ylabel('Number of Improvements', color='#A23B72')
        ax1_twin.tick_params(axis='y', labelcolor='#A23B72')
        
        # Alternation frequency sensitivity
        freqs = [r['frequency'] for r in freq_results]
        costs_freq = [r['cost'] for r in freq_results]
        improvements_freq = [r['improvements'] for r in freq_results]
        
        ax2.plot(freqs, costs_freq, 'o-', linewidth=2, markersize=8, color='#2E86AB')
        ax2.set_xlabel('Alternation Frequency')
        ax2.set_ylabel('Final Cost', color='#2E86AB')
        ax2.tick_params(axis='y', labelcolor='#2E86AB')
        ax2.set_title('Alternation Frequency Sensitivity')
        ax2.grid(True, alpha=0.3)
        
        ax2_twin = ax2.twinx()
        ax2_twin.plot(freqs, improvements_freq, 's--', linewidth=2, markersize=8, color='#A23B72')
        ax2_twin.set_ylabel('Number of Improvements', color='#A23B72')
        ax2_twin.tick_params(axis='y', labelcolor='#A23B72')
        
        plt.tight_layout()
        plt.show()
        
        # Parameter recommendations
        print("\n=== PARAMETER RECOMMENDATIONS ===")
        
        best_cooling = min(cooling_results, key=lambda x: x['cost'])
        best_frequency = min(freq_results, key=lambda x: x['cost'])
        
        print(f"Best cooling rate: {best_cooling['cooling_rate']:.2f} (Cost: {best_cooling['cost']:.1f})")
        print(f"Best alternation frequency: {best_frequency['frequency']} (Cost: {best_frequency['cost']:.1f})")
        
        print("\nGeneral guidelines:")
        print("- Higher cooling rates (0.95-0.97) provide better balance")
        print("- Moderate alternation frequencies (100-200) work well")
        print("- Too frequent alternation may limit each phase's effectiveness")
        print("- Too infrequent alternation may reduce hybrid benefits")
    
    # Perform parameter tuning analysis
    parameter_tuning_analysis(hybrid_instance)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'hybrid_instance' is not defined...]

### Why this Tier exists vs earlier Tiers
This Tier 3 introduces a **Hybrid Metaheuristic** that combines the complementary strengths of Simulated Annealing (SA) and Variable Neighborhood Search (VNS). This hybrid approach addresses limitations of both individual methods while leveraging their unique capabilities for superior performance.

**Advantages vs Tier 1 (CSP):**
- **Dramatically better scalability**: Handles large instances efficiently
- **Hybrid robustness**: Combines global exploration with local intensification
- **Adaptive search**: Dynamic alternation between exploration and exploitation
- **Practical performance**: Suitable for real-world decision timeframes

**Advantages vs Tier 2 (Cross Entropy):**
- **Deterministic local search**: VNS provides thorough local optimization
- **Temperature control**: SA offers systematic escape from local optima
- **Neighborhood diversity**: Multiple neighborhood structures for comprehensive search
- **Phase alternation**: Dynamic balance between diversification and intensification

**Synergistic benefits of the hybrid approach:**
- **SA strengths**: Global exploration, probabilistic acceptance, temperature-controlled search
- **VNS strengths**: Systematic local search, multiple neighborhood structures, thorough intensification
- **Hybrid advantages**: Best of both worlds with adaptive phase alternation

**Disadvantages vs earlier Tiers:**
- **Parameter complexity**: More parameters to tune than individual methods
- **Computational overhead**: Two algorithms require more resources
- **Implementation complexity**: Hybrid coordination adds sophistication

**When to use this Tier:**
- Medium to large instances requiring high-quality solutions
- Problems where both global exploration and local refinement are important
- Situations where single metaheuristics get stuck in local optima
- Applications requiring robust performance across diverse problem instances

### Pros / Cons Summary
**Pros:**
✓ Superior solution quality through hybridization
✓ Effective balance of exploration and exploitation
✓ Robust performance across problem variations
✓ Systematic local search with VNS neighborhood structures
✓ Global optimization with SA temperature control
✓ Adaptive phase alternation for dynamic search behavior
✓ Strong theoretical foundation from both component algorithms

**Cons:**
✗ Higher parameter complexity
✗ Increased computational requirements
✗ More sophisticated implementation
✗ Requires careful parameter tuning for optimal performance
✗ May be overkill for simple problem instances

The Hybrid SA-VNS approach represents the state-of-the-art in metaheuristic design for facility location problems, providing exceptional solution quality through the intelligent combination of complementary search strategies.